Retrieve [research topics from OpenAlex](https://openalex.org/works?page=1&filter=cites%3Aw2015197254) for all works that cite the original pymatgen paper.

Then use LLM to organize topics into a mindmap.

Note:
- You would need to get `OPENAI_API_KEY` to use OpenAI API, see https://openai.com/api/.

In [ ]:
!uv pip install requests openai pyyaml matplotlib

NUM_OF_MAIN_TOPICS: int = 5
NUM_OF_SUB_TOPICS: int = 4

Using Python 3.12.11 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Audited 4 packages in 5ms


In [ ]:
# Step 1: Get topics from OpenAlex for 1st pymatgen paper

from collections import Counter

import requests

params = {"filter": "cites:w2015197254", "group_by": "primary_topic.id", "per_page": 50}
response = requests.get(
    "https://api.openalex.org/works", params=params, timeout=10
).json()

topic_counter = Counter(
    {
        group.get("key_display_name", "None"): group["count"]
        for group in response.get("group_by")
        if group["count"] >= 10  # Count for top 50 topics is around 10
    }
)

print("Topics from OpenAlex:")
for topic, count in topic_counter.items():
    print(f"  {topic:<60}{count}")

Topics from OpenAlex:
  Machine Learning in Materials Science                       1172
  Advancements in Battery Materials                           325
  Advanced Battery Materials and Technologies                 325
  Perovskite Materials and Applications                       110
  Electrocatalysts for Energy Conversion                      73
  Metal-Organic Frameworks: Synthesis and Applications        73
  2D Materials and Applications                               68
  Electronic and Structural Properties of Oxides              66
  Advanced Thermoelectric Materials and Devices               55
  MXene and MAX Phase Materials                               49
  Synthesis and Properties of Inorganic Cluster Compounds     41
  X-ray Diffraction in Crystallography                        38
  Chalcogenide Semiconductor Thin Films                       36
  Advancements in Solid Oxide Fuel Cells                      32
  High Entropy Alloys Studies                                 3

In [ ]:
# Step 2: Use LLM to group topics

import re

from openai import OpenAI

client = OpenAI()  # https://platform.openai.com/docs/overview

topic_list = "\n".join(f"- {topic} ({count})" for topic, count in topic_counter.items())

prompt = f"""
The following research topics are from papers that cite pymatgen, with each number indicating the count of citing works in that area.

Please organize them into {NUM_OF_MAIN_TOPICS} thematic branches that represent broader areas of materials research (e.g., batteries, machine learning, catalysis, ...).

Return output in a plain text tree structure (with indentation) without any header or metadata, e.g.:
    Topic_0:
        subtopic_0 (count_0)
        subtopic_1 (count_1)

    Topic_1:
        subtopic_0 (count_0)

Topics to work on today:
    {topic_list}
"""

chat_response = client.responses.create(
    model="gpt-4.1",
    input=prompt,
)

# print(chat_response.output_text)


def parse_mindmap(text: str) -> dict[str, list[tuple[str, int]]]:
    """
    Parse LLM response to machine-readable dict for plotter.
    """
    mindmap: dict[str, list[tuple[str, int]]] = {}
    current_topic = None

    for line in text.strip().splitlines():
        line = line.rstrip()

        if not line.strip():
            continue

        if not line.startswith(" "):  # Top-level topic
            current_topic = line.strip().removesuffix(":")
            mindmap[current_topic] = []

        elif current_topic:
            subtopic_line = line.strip()

            # Extract the (name, count)
            if match := re.match(r"^(.*)\s+\((\d+)\)$", subtopic_line):
                name = match[1].strip()
                count = int(match[2])
                mindmap[current_topic].append((name, count))
            else:
                raise ValueError(f"Cannot extract info from {subtopic_line=}")

    return mindmap


mindmap_dict: dict[str, list[tuple[str, int]]] = parse_mindmap(
    chat_response.output_text
)

# Sort main topics by the sum of subtopic counts
sorted_main_topics = sorted(
    mindmap_dict.items(),
    key=lambda item: sum(count for _, count in item[1]),
    reverse=True,
)

# Limit max number of main/sub topics
mindmap_dict = {
    main_topic: sorted(subtopics, key=lambda x: x[1], reverse=True)[:NUM_OF_SUB_TOPICS]
    for main_topic, subtopics in sorted_main_topics[:NUM_OF_MAIN_TOPICS]
}

for main_topic, subtopics in mindmap_dict.items():
    print(f"{main_topic} (total {sum(c for _, c in subtopics)}):")
    for name, count in subtopics:
        print(f"    - {name} ({count})")

Machine Learning and Computational Methods (total 1200):
    - Machine Learning in Materials Science (1172)
    - Scientific Computing and Data Management (14)
    - Computational Drug Discovery Methods (14)
Energy Storage and Conversion (total 709):
    - Advancements in Battery Materials (325)
    - Advanced Battery Materials and Technologies (325)
    - Advancements in Solid Oxide Fuel Cells (32)
    - Ammonia Synthesis and Nitrogen Reduction (27)
Electronic, Magnetic, and Quantum Materials (total 293):
    - Perovskite Materials and Applications (110)
    - 2D Materials and Applications (68)
    - Electronic and Structural Properties of Oxides (66)
    - MXene and MAX Phase Materials (49)
Synthesis, Characterization, and Structural Properties (total 175):
    - Advanced Thermoelectric Materials and Devices (55)
    - Synthesis and Properties of Inorganic Cluster Compounds (41)
    - Synthesis and Properties of Inorganic Cluster Compounds (41)
    - X-ray Diffraction in Crystallogra

In [ ]:
# Step 3: Export LLM summarized topic data to YAML for Typst with matplotlib colormap
import yaml
import matplotlib as mpl

START_ANGLES = [45, 10, -60, 200, 120]
assert len(START_ANGLES) == NUM_OF_MAIN_TOPICS

# Preserve current sorted/truncated order from Step 2
ordered_main = list(mindmap_dict.items())
if len(ordered_main) != NUM_OF_MAIN_TOPICS:
    raise ValueError(f"Unexpected number of topics, expect {NUM_OF_MAIN_TOPICS}")

# --- Collect all child values for global min/max ---
all_values = [count for _, subs in ordered_main for _, count in subs]
vmin, vmax = min(all_values), max(all_values)

# --- Choose colormap & normalizer ---
cmap = mpl.colormaps["viridis"]  # TODO: pick cmap
norm = mpl.colors.LogNorm(vmin=max(vmin, 1e-6), vmax=vmax)  # log scale


def value_to_hex(val: float) -> str:
    """Map a numeric value to a hex color."""
    rgba = cmap(norm(val))
    return mpl.colors.to_hex(rgba, keep_alpha=False)


# --- Build YAML data ---
branches = []
for i, (main_title, subtopics) in enumerate(ordered_main):
    children = [
        {"title": sub_title, "value": int(count), "color": value_to_hex(count)}
        for sub_title, count in subtopics
    ]
    # Parent color from sum of children
    parent_val = sum([c["value"] for c in children])
    branches.append(
        {
            "title": main_title,
            "color": value_to_hex(parent_val),
            "start_angle_deg": START_ANGLES[i],
            "children": children,
        }
    )

data = {"title": "pymatgen", "branches": branches}

with open("llm_summarized_topics.yml", "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

In [ ]:
# Step 4: Compile Typst to SVG
!typst compile mindmap.typ mindmap.svg